In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## 6.2 Dataset Preparation

In [14]:
import urllib.request
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"


def download_and_unzip_spam_data(
        url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download "
              "and extraction."
        )
        return

    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"File downloaded and saved as {data_file_path}")

download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

sms_spam_collection\SMSSpamCollection.tsv already exists. Skipping download and extraction.


In [15]:
import pandas as pd
df = pd.read_csv(
    data_file_path, sep="\t", header=None, names=["Label", "Text"]
)
df

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [16]:
df.head()

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [17]:
print(df["Label"].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


In [18]:
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0]
    ham_subset = df[df["Label"] == "ham"].sample(
        num_spam, random_state=123
    )
    balanced_df = pd.concat([
        ham_subset, df[df["Label"] == "spam"]
    ])
    return balanced_df

balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


In [19]:
def create_unbalanced_dataset(df):
    ham_subset = df[df["Label"] == "ham"]
    spam_subset = df[df["Label"] == "spam"]
    unbalanced_df = pd.concat([ham_subset, spam_subset])

    return unbalanced_df

unbalanced_df = create_unbalanced_dataset(df)
print(unbalanced_df["Label"].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


In [20]:
def random_split(df, train_frac, validation_frac):

    df = df.sample(
        frac=1, random_state=123
    ).reset_index(drop=True)
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)


    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(
    balanced_df, 0.7, 0.1)

In [21]:
train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

## 6.3 Create Data Loaders

## 6.4.A Focal Loss Implementation

Link to the focal loss paper: https://github.com/itakurah/focal-loss-pytorch

In [34]:
y = torch.randn((8,3))
y

tensor([[-0.2254, -0.4706, -0.6391],
        [-0.9753, -0.7379,  0.2083],
        [ 0.8994,  2.8469,  0.4718],
        [-0.7046,  0.6493,  0.4784],
        [-0.2404,  0.5932, -0.7962],
        [ 0.8679,  2.1765,  0.3381],
        [ 0.4212, -0.1230,  0.6628],
        [ 2.2454, -0.3741,  0.3321]])

In [36]:
rows = [1,2]
cols = [0,1]

y[rows,cols]

tensor([-0.9753,  2.8469])

In [ ]:
class FocalLoss(nn.Module):
    """
    Multi-class Focal Loss for per-token classification.

        FL(p_t) = - alpha_t * (1 - p_t)**gamma * log(p_t)

    Inputs are token-level logits of shape (batch_size, seq_len, num_classes)
    and targets are class indices of shape (batch_size, seq_len). Both are
    flattened to (batch_size * seq_len, ...) before the focal-loss math.

    Setting alpha proportional to *inverse class frequency* up-weights minority
    classes so that rare-but-hard tokens drive the gradient.
    """

    def __init__(self, gamma=2.0, alpha=None):
        """
        :param gamma: focusing parameter; down-weights easy (high p_t) examples.
        :param alpha: class balancing factor. One of:
            - None: no class weighting (alpha_t = 1 for every token).
            - float: binary case; weight for class 1, (1 - alpha) for class 0.
            - 1-D tensor of shape (num_classes,): per-class weights.
              Use FocalLoss.alpha_from_counts(counts) for inverse-frequency weights.
        """
        super().__init__()
        self.gamma = gamma
        if torch.is_tensor(alpha):
            # Register as buffer so it moves with .to(device) and saves with the module.
            self.register_buffer("alpha", alpha.float())
        else:
            self.alpha = alpha  # None or float scalar

    @staticmethod
    def alpha_from_counts(counts):
        """
        Inverse-frequency class weights (sklearn "balanced" convention):

            alpha[c] = N_total / (num_classes * count[c])

        Properties:
          - For a perfectly balanced dataset every alpha[c] == 1.
          - Minority classes get alpha > 1, majority classes get alpha < 1.
          - sum(count[c] * alpha[c]) == N_total, so each class contributes
            an equal *expected* weight to the loss.
        """
        counts = torch.as_tensor(counts, dtype=torch.float32)
        num_classes = counts.numel()
        n_total = counts.sum()
        return n_total / (num_classes * counts)

    def forward(self, inputs, targets):
        """
        :param inputs:  logits of shape (batch_size, seq_len, num_classes)
        :param targets: class indices of shape (batch_size, seq_len)
        """
        num_classes = inputs.shape[-1]
        # Collapse the (batch, seq) axes so every token becomes an independent sample.
        inputs_flat  = inputs.reshape(-1, num_classes)    # (B*S, C)
        targets_flat = targets.reshape(-1)                # (B*S,)

        # Per-token cross-entropy = -log p_t for the true class.
        ce = F.cross_entropy(inputs_flat, targets_flat, reduction="none")
        # Recover p_t = exp(-ce). The original code did exp(ce), which gives 1/p_t.
        p_t = torch.exp(-ce)

        # Per-token alpha_t
        if self.alpha is None:
            alpha_t = 1.0
        elif torch.is_tensor(self.alpha):
            alpha_t = self.alpha[targets_flat]
        else:
            # Binary scalar: alpha for class 1, (1 - alpha) for class 0.
            alpha_t = torch.where(
                targets_flat == 1,
                torch.full_like(ce, self.alpha),
                torch.full_like(ce, 1.0 - self.alpha),
            )

        # Focal term * NLL. ce already equals -log(p_t), so no leading minus.
        loss = alpha_t * (1.0 - p_t) ** self.gamma * ce
        return loss.mean()

### Using inverse-frequency alpha on the SMS spam dataset

The full corpus has **4825 ham** vs **747 spam** messages. Plugging those into
`FocalLoss.alpha_from_counts` gives:

- `alpha[ham]  = 5572 / (2 * 4825) ≈ 0.577`  (majority → down-weighted)
- `alpha[spam] = 5572 / (2 *  747) ≈ 3.730`  (minority → up-weighted)

So a misclassified spam token contributes ~6.5× more gradient than a misclassified
ham token, which is exactly the imbalance ratio. For the balanced training split
(747 ham, 747 spam) both weights are 1.0 — i.e. focal loss with this alpha
gracefully degenerates to the un-weighted version when the data is already balanced.

In [ ]:
# Class counts straight from the unbalanced SMS spam corpus
# (index 0 = ham, index 1 = spam, matching the eventual label encoding)
class_counts = [
    int((df["Label"] == "ham").sum()),
    int((df["Label"] == "spam").sum()),
]
print("class counts:", class_counts)

alpha = FocalLoss.alpha_from_counts(class_counts)
print("inverse-frequency alpha:", alpha)

criterion = FocalLoss(gamma=2.0, alpha=alpha)
print(criterion)

# Smoke test on dummy per-token logits/targets: (B, S, C) and (B, S)
torch.manual_seed(0)
batch_size, seq_len, num_classes = 4, 16, 2
logits  = torch.randn(batch_size, seq_len, num_classes, requires_grad=True)
targets = torch.randint(0, num_classes, (batch_size, seq_len))
loss = criterion(logits, targets)
loss.backward()
print(f"focal loss = {loss.item():.4f}, grad ok: {logits.grad is not None}")